# Module 0 - Frozen Canonical Dataset

The canonical dataset has been frozen as D0.csv. In accordance with the governing research protocol, the remaining pipeline reads from this file only and does not recreate or modify it.


# Module 1

In [2]:
import numpy as np
import pandas as pd

D0 = pd.read_csv("D0.csv")

canonical_columns = [
    "Date", "Materials", "Energy", "Financials", "Industrials", "Technology",
    "Consumer_Staples", "Utilities", "Healthcare", "Consumer_Discretionary",
    "PPI_Energy", "PPI_Metals", "PPI_Chemicals", "PPI_Food", "RiskFreeRate",
]
assert D0.columns.tolist() == canonical_columns, "D0 schema differs from the frozen canonical schema."

D0["Date"] = pd.to_datetime(D0["Date"])
sector_columns = canonical_columns[1:10]
ppi_columns = canonical_columns[10:14]

assert D0.shape == (194, 15)
assert D0["Date"].duplicated().sum() == 0
assert D0.duplicated().sum() == 0
assert D0["Date"].is_monotonic_increasing
assert D0["Date"].reset_index(drop=True).equals(pd.Series(pd.date_range(D0["Date"].min(), D0["Date"].max(), freq="ME")))
assert D0.isna().sum().sum() == 0
assert not np.isinf(D0.drop(columns="Date").to_numpy(dtype=float)).any()
assert (D0[sector_columns] > 0).all().all()
assert (D0[ppi_columns] > 0).all().all()

prices = D0[sector_columns]
log_returns = np.log(prices).diff()
# RiskFreeRate is an annualised nominal percentage rate (monthly-compounding convention); divide by 1200 before the log conversion.
monthly_log_risk_free_rate = np.log1p(D0["RiskFreeRate"] / 1200)
excess_log_returns = log_returns.sub(monthly_log_risk_free_rate, axis=0)
cross_sectional_average_return = log_returns.mean(axis=1)
cross_sectional_average_excess_return = excess_log_returns.mean(axis=1)
relative_log_returns = log_returns.sub(cross_sectional_average_return, axis=0)
momentum_12m = np.log(prices / prices.shift(12))
realized_volatility_12m = excess_log_returns.rolling(12, min_periods=12).std(ddof=1)
rolling_mean_return_6m = excess_log_returns.rolling(6, min_periods=6).mean()
assert realized_volatility_12m.dropna().gt(0).all().all()

# Own-history standardisation: how unusual the current excess return is relative to the sector's own trailing distribution.
rolling_mean_excess_12m = excess_log_returns.rolling(12, min_periods=12).mean()
rolling_zscore_excess_12m = (excess_log_returns - rolling_mean_excess_12m) / realized_volatility_12m

# Cross-sectional standardisation: relative performance scaled by the contemporaneous cross-sector dispersion regime.
cross_sectional_dispersion = log_returns.std(axis=1, ddof=1)
assert cross_sectional_dispersion.dropna().gt(0).all()
relative_zscore = relative_log_returns.div(cross_sectional_dispersion, axis=0)

ppi_log_inflation_1m = np.log(D0[ppi_columns]).diff()
ppi_log_inflation_12m = np.log(D0[ppi_columns] / D0[ppi_columns].shift(12))

# Discount-rate shock: level is already embedded in excess returns, so only the change is informative here.
risk_free_rate_change_1m = D0["RiskFreeRate"].diff().rename("RiskFreeRateChange1M")
risk_free_rate_change_12m = (D0["RiskFreeRate"] - D0["RiskFreeRate"].shift(12)).rename("RiskFreeRateChange12M")

def prefixed(frame, prefix):
    return frame.rename(columns={column: f"{prefix}{column}" for column in frame.columns})

D1 = pd.concat([
    D0[["Date"]],
    prefixed(log_returns, "LogReturn_"),
    prefixed(excess_log_returns, "ExcessLogReturn_"),
    prefixed(relative_log_returns, "RelativeLogReturn_"),
    prefixed(momentum_12m, "Momentum12M_"),
    prefixed(realized_volatility_12m, "Volatility12M_"),
    prefixed(rolling_mean_return_6m, "RollingMeanReturn6M_"),
    prefixed(rolling_zscore_excess_12m, "ExcessReturnZScore12M_"),
    prefixed(relative_zscore, "RelativeZScore_"),
    prefixed(ppi_log_inflation_1m, "PPIInflation1M_"),
    prefixed(ppi_log_inflation_12m, "PPIInflation12M_"),
    D0[["RiskFreeRate"]],
    risk_free_rate_change_1m,
    risk_free_rate_change_12m,
    cross_sectional_average_return.rename("CrossSectionalAverageLogReturn"),
    cross_sectional_average_excess_return.rename("CrossSectionalAverageExcessLogReturn"),
], axis=1).dropna().reset_index(drop=True)

feature_blocks = {
    "log_return": [f"LogReturn_{sector}" for sector in sector_columns],
    "excess_return": [f"ExcessLogReturn_{sector}" for sector in sector_columns],
    "relative_return": [f"RelativeLogReturn_{sector}" for sector in sector_columns],
    "momentum": [f"Momentum12M_{sector}" for sector in sector_columns],
    "volatility": [f"Volatility12M_{sector}" for sector in sector_columns],
    "rolling_mean_return": [f"RollingMeanReturn6M_{sector}" for sector in sector_columns],
    "excess_return_zscore": [f"ExcessReturnZScore12M_{sector}" for sector in sector_columns],
    "relative_zscore": [f"RelativeZScore_{sector}" for sector in sector_columns],
    "ppi_1m": [f"PPIInflation1M_{ppi}" for ppi in ppi_columns],
    "ppi_12m": [f"PPIInflation12M_{ppi}" for ppi in ppi_columns],
    "risk_free_rate": ["RiskFreeRateChange1M", "RiskFreeRateChange12M"],
}
feature_block_sizes = {name: len(columns) for name, columns in feature_blocks.items()}

# Global/aggregate features describe the whole cross-section or the discount rate itself, so they are
# deliberately not members of any per-sector or per-PPI-series engineered block.
aggregate_features = [
    "RiskFreeRate",
    "CrossSectionalAverageLogReturn",
    "CrossSectionalAverageExcessLogReturn",
]
assert len(aggregate_features) == 3

engineered_block_feature_count = sum(feature_block_sizes.values())
aggregate_feature_count = len(aggregate_features)
total_numeric_features = int(D1.drop(columns="Date").shape[1])

assert D1.shape == (182, 86)
assert D1["Date"].duplicated().sum() == 0
assert D1.duplicated().sum() == 0
assert D1.isna().sum().sum() == 0
assert not np.isinf(D1.drop(columns="Date").to_numpy(dtype=float)).any()
assert D1["Date"].is_monotonic_increasing
assert D1["Date"].reset_index(drop=True).equals(pd.Series(pd.date_range(D1["Date"].min(), D1["Date"].max(), freq="ME")))
assert all(
    len(columns) == len(sector_columns)
    for name, columns in feature_blocks.items()
    if name not in {"ppi_1m", "ppi_12m", "risk_free_rate"}
)
assert all(len(feature_blocks[name]) == len(ppi_columns) for name in {"ppi_1m", "ppi_12m"})
assert len(feature_blocks["risk_free_rate"]) == 2
assert engineered_block_feature_count == 82
assert engineered_block_feature_count + aggregate_feature_count == total_numeric_features

D1.to_csv("D1.csv", index=False)

mean_absolute_excess_return_zscore = float(
    D1[feature_blocks["excess_return_zscore"]].to_numpy().__abs__().mean()
)
mean_absolute_relative_zscore = float(
    D1[feature_blocks["relative_zscore"]].to_numpy().__abs__().mean()
)
mean_absolute_risk_free_rate_change_1m = float(D1["RiskFreeRateChange1M"].abs().mean())
mean_absolute_risk_free_rate_change_12m = float(D1["RiskFreeRateChange12M"].abs().mean())

module_1_validation = {
    "D0_schema_valid": True,
    "D1_shape": D1.shape,
    "date_range": f"{D1['Date'].min().date()} to {D1['Date'].max().date()}",
    "duplicate_dates": int(D1["Date"].duplicated().sum()),
    "duplicate_rows": int(D1.duplicated().sum()),
    "missing_values": int(D1.isna().sum().sum()),
    "infinite_values": int(np.isinf(D1.drop(columns="Date").to_numpy(dtype=float)).sum()),
    "monthly_continuity": True,
    "chronological_order": True,
    "matrix_compatible": True,
    "engineered_feature_blocks": list(feature_blocks.keys()),
    "feature_block_sizes": feature_block_sizes,
    "engineered_block_feature_count": engineered_block_feature_count,
    "aggregate_features": aggregate_features,
    "aggregate_feature_count": aggregate_feature_count,
    "total_numeric_features": total_numeric_features,
    "mean_absolute_excess_return_zscore": mean_absolute_excess_return_zscore,
    "mean_absolute_relative_zscore": mean_absolute_relative_zscore,
    "mean_absolute_risk_free_rate_change_1m": mean_absolute_risk_free_rate_change_1m,
    "mean_absolute_risk_free_rate_change_12m": mean_absolute_risk_free_rate_change_12m,
    "look_ahead_bias": "No negative shifts, centered windows, or future observations were used.",
    "data_leakage": "No target variables or full-sample standardisation were created; scaling is deferred to walk-forward modelling.",
}
module_1_validation



{'D0_schema_valid': True,
 'D1_shape': (182, 86),
 'date_range': '2011-01-31 to 2026-02-28',
 'duplicate_dates': 0,
 'duplicate_rows': 0,
 'missing_values': 0,
 'infinite_values': 0,
 'monthly_continuity': True,
 'chronological_order': True,
 'matrix_compatible': True,
 'engineered_feature_blocks': ['log_return',
  'excess_return',
  'relative_return',
  'momentum',
  'volatility',
  'rolling_mean_return',
  'excess_return_zscore',
  'relative_zscore',
  'ppi_1m',
  'ppi_12m',
  'risk_free_rate'],
 'feature_block_sizes': {'log_return': 9,
  'excess_return': 9,
  'relative_return': 9,
  'momentum': 9,
  'volatility': 9,
  'rolling_mean_return': 9,
  'excess_return_zscore': 9,
  'relative_zscore': 9,
  'ppi_1m': 4,
  'ppi_12m': 4,
  'risk_free_rate': 2},
 'engineered_block_feature_count': 82,
 'aggregate_features': ['RiskFreeRate',
  'CrossSectionalAverageLogReturn',
  'CrossSectionalAverageExcessLogReturn'],
 'aggregate_feature_count': 3,
 'total_numeric_features': 85,
 'mean_absolute_e

# Module 2

In [7]:
from scipy.optimize import linear_sum_assignment

D1 = pd.read_csv("D1.csv")
D1["Date"] = pd.to_datetime(D1["Date"])

assert D1.shape == (182, 86)
assert D1["Date"].duplicated().sum() == 0
assert D1.duplicated().sum() == 0
assert D1.isna().sum().sum() == 0
assert not np.isinf(D1.drop(columns="Date").to_numpy(dtype=float)).any()
assert D1["Date"].is_monotonic_increasing
assert D1["Date"].reset_index(drop=True).equals(
    pd.Series(pd.date_range(D1["Date"].min(), D1["Date"].max(), freq="ME"))
)

sector_columns = [
    "Materials", "Energy", "Financials", "Industrials", "Technology",
    "Consumer_Staples", "Utilities", "Healthcare", "Consumer_Discretionary",
]
excess_columns = [f"ExcessLogReturn_{sector}" for sector in sector_columns]
macro_columns = [
    "PPIInflation1M_PPI_Energy", "PPIInflation1M_PPI_Metals",
    "PPIInflation1M_PPI_Chemicals", "PPIInflation1M_PPI_Food",
    "PPIInflation12M_PPI_Energy", "PPIInflation12M_PPI_Metals",
    "PPIInflation12M_PPI_Chemicals", "PPIInflation12M_PPI_Food",
    "RiskFreeRate",
]

n_sectors = len(sector_columns)
factor_window = 36
regression_window = 36
n_factors = 2
ridge_penalty = 1.0

excess_returns = D1[excess_columns].to_numpy(dtype=float)
macro_data = D1[macro_columns].to_numpy(dtype=float)
n_obs = len(D1)

predictors = np.full((n_obs, len(macro_columns) + n_factors), np.nan)
factor_scores = np.full((n_obs, n_factors), np.nan)
factor_loadings = np.full((n_obs, n_sectors, n_factors), np.nan)
previous_loadings = None

for t in range(factor_window, n_obs):
    factor_history = excess_returns[t - factor_window:t]
    factor_mean = factor_history.mean(axis=0)
    factor_scale = factor_history.std(axis=0, ddof=1)
    assert np.all(factor_scale > 0)
    standardized_history = (factor_history - factor_mean) / factor_scale
    _, _, right_singular_vectors = np.linalg.svd(standardized_history, full_matrices=False)
    raw_loadings = right_singular_vectors[:n_factors].T

    if previous_loadings is None:
        loadings = raw_loadings.copy()
        for factor in range(n_factors):
            dominant_loading = loadings[np.argmax(np.abs(loadings[:, factor])), factor]
            if dominant_loading < 0:
                loadings[:, factor] *= -1
    else:
        # First, solve the factor-matching assignment exactly with the Hungarian algorithm.
        # This maximizes total absolute overlap in polynomial time and replaces the previous
        # factorial-time brute-force permutation search.
        similarity = previous_loadings.T @ raw_loadings
        row_index, column_index = linear_sum_assignment(-np.abs(similarity))
        best_permutation = np.empty(n_factors, dtype=int)
        best_permutation[row_index] = column_index
        permuted_loadings = raw_loadings[:, best_permutation]

        # Then align factor signs to avoid arbitrary reflections across windows.
        for factor in range(n_factors):
            if np.dot(permuted_loadings[:, factor], previous_loadings[:, factor]) < 0:
                permuted_loadings[:, factor] *= -1

        # Finally, apply an orthogonal Procrustes alignment. This computes the
        # orthogonal rotation that minimizes Frobenius distance to the previous basis,
        # making the rolling latent factor coordinate system evolve smoothly over time.
        alignment_cross_moment = permuted_loadings.T @ previous_loadings
        u_matrix, _, v_t_matrix = np.linalg.svd(alignment_cross_moment, full_matrices=False)
        orthogonal_rotation = u_matrix @ v_t_matrix
        loadings = permuted_loadings @ orthogonal_rotation

    assert np.allclose(loadings.T @ loadings, np.eye(n_factors), atol=1e-10)

    previous_loadings = loadings.copy()
    previous_excess_return = (excess_returns[t - 1] - factor_mean) / factor_scale
    factor_scores[t] = previous_excess_return @ loadings
    factor_loadings[t] = loadings
    predictors[t] = np.concatenate((macro_data[t - 1], factor_scores[t]))

equilibrium_forecasts = np.full((n_obs, n_sectors), np.nan)

for t in range(factor_window + regression_window, n_obs):
    training_indices = np.arange(t - regression_window, t)
    X_train = predictors[training_indices]
    y_train = excess_returns[training_indices]

    assert np.isfinite(X_train).all()
    assert np.isfinite(y_train).all()

    X_mean = X_train.mean(axis=0)
    X_scale = X_train.std(axis=0, ddof=1)
    assert np.all(X_scale > 0)

    X_train_standardized = (X_train - X_mean) / X_scale
    x_forecast_standardized = (predictors[t] - X_mean) / X_scale
    y_mean = y_train.mean(axis=0)

    ridge_system = X_train_standardized.T @ X_train_standardized
    ridge_system += ridge_penalty * np.eye(ridge_system.shape[0])
    ridge_coefficients = np.linalg.solve(
        ridge_system,
        X_train_standardized.T @ (y_train - y_mean),
    )
    equilibrium_forecasts[t] = y_mean + x_forecast_standardized @ ridge_coefficients

valid_rows = np.isfinite(equilibrium_forecasts).all(axis=1)
assert valid_rows.sum() == 110

equilibrium_columns = [f"EquilibriumExcessReturn_{sector}" for sector in sector_columns]
actual_columns = [f"ActualExcessReturn_{sector}" for sector in sector_columns]
residual_columns = [f"EquilibriumResidual_{sector}" for sector in sector_columns]

D2 = pd.concat(
    [
        D1.loc[valid_rows, ["Date"]].reset_index(drop=True),
        pd.DataFrame(excess_returns[valid_rows], columns=actual_columns),
        pd.DataFrame(equilibrium_forecasts[valid_rows], columns=equilibrium_columns),
        pd.DataFrame(
            excess_returns[valid_rows] - equilibrium_forecasts[valid_rows],
            columns=residual_columns,
        ),
        pd.DataFrame(factor_scores[valid_rows], columns=["LatentFactor1", "LatentFactor2"]),
    ],
    axis=1,
)

loading_columns = [
    f"Factor{factor + 1}Loading_{sector}"
    for factor in range(n_factors)
    for sector in sector_columns
]
factor_loadings_output = pd.concat(
    [
        D1.loc[valid_rows, ["Date"]].reset_index(drop=True),
        pd.DataFrame(
            factor_loadings[valid_rows].transpose(0, 2, 1).reshape(valid_rows.sum(), -1),
            columns=loading_columns,
        ),
    ],
    axis=1,
)

assert D2.shape == (110, 30)
assert D2["Date"].duplicated().sum() == 0
assert D2.duplicated().sum() == 0
assert D2.isna().sum().sum() == 0
assert not np.isinf(D2.drop(columns="Date").to_numpy(dtype=float)).any()
assert D2["Date"].is_monotonic_increasing
assert D2["Date"].reset_index(drop=True).equals(
    pd.Series(pd.date_range(D2["Date"].min(), D2["Date"].max(), freq="ME"))
)
assert np.allclose(
    D2[actual_columns].to_numpy() - D2[equilibrium_columns].to_numpy(),
    D2[residual_columns].to_numpy(),
)
assert factor_loadings_output.shape == (110, 19)
assert factor_loadings_output["Date"].equals(D2["Date"])

D2.to_csv("D2.csv", index=False)
factor_loadings_output.to_csv("equilibrium_factor_loadings.csv", index=False)

actual_matrix = D2[actual_columns].to_numpy()
equilibrium_matrix = D2[equilibrium_columns].to_numpy()
residual_matrix = D2[residual_columns].to_numpy()
out_of_sample_r2_zero = 1 - np.sum(residual_matrix ** 2) / np.sum(actual_matrix ** 2)

module_2_validation = {
    "D2_shape": D2.shape,
    "date_range": f"{D2['Date'].min().date()} to {D2['Date'].max().date()}",
    "duplicate_dates": int(D2["Date"].duplicated().sum()),
    "duplicate_rows": int(D2.duplicated().sum()),
    "missing_values": int(D2.isna().sum().sum()),
    "infinite_values": int(np.isinf(D2.drop(columns="Date").to_numpy(dtype=float)).sum()),
    "monthly_continuity": True,
    "chronological_order": True,
    "matrix_compatibility": True,
    "residual_identity": True,
    "out_of_sample_r2_against_zero": float(out_of_sample_r2_zero),
    "mean_absolute_equilibrium_residual": float(np.mean(np.abs(residual_matrix))),
    "look_ahead_bias": "Each forecast uses only lagged macro data, a factor window ending at t-1, and a regression window ending at t-1.",
    "data_leakage": "PCA, feature scaling, and ridge coefficients are re-estimated within each rolling historical window.",
}
module_2_validation



{'D2_shape': (110, 30),
 'date_range': '2017-01-31 to 2026-02-28',
 'duplicate_dates': 0,
 'duplicate_rows': 0,
 'missing_values': 0,
 'infinite_values': 0,
 'monthly_continuity': True,
 'chronological_order': True,
 'matrix_compatibility': True,
 'residual_identity': True,
 'out_of_sample_r2_against_zero': -0.7064178812244784,
 'mean_absolute_equilibrium_residual': 0.05386559692229284,
 'look_ahead_bias': 'Each forecast uses only lagged macro data, a factor window ending at t-1, and a regression window ending at t-1.',
 'data_leakage': 'PCA, feature scaling, and ridge coefficients are re-estimated within each rolling historical window.'}

# Module 3

In [10]:
D2 = pd.read_csv("D2.csv")
D2["Date"] = pd.to_datetime(D2["Date"])

assert D2.shape == (110, 30)
assert D2["Date"].duplicated().sum() == 0
assert D2.duplicated().sum() == 0
assert D2.isna().sum().sum() == 0
assert not np.isinf(D2.drop(columns="Date").to_numpy(dtype=float)).any()
assert D2["Date"].is_monotonic_increasing
assert D2["Date"].reset_index(drop=True).equals(
    pd.Series(pd.date_range(D2["Date"].min(), D2["Date"].max(), freq="ME"))
)

sector_columns = [
    "Materials", "Energy", "Financials", "Industrials", "Technology",
    "Consumer_Staples", "Utilities", "Healthcare", "Consumer_Discretionary",
]
residual_columns = [f"EquilibriumResidual_{sector}" for sector in sector_columns]
residuals = D2[residual_columns].to_numpy(dtype=float)

calibration_period = 24
rolling_parameter_window = 36
n_obs, n_sectors = residuals.shape
assert n_obs > calibration_period

state_estimates = np.full((n_obs, n_sectors), np.nan)
standardized_states = np.full((n_obs, n_sectors), np.nan)
state_changes = np.full((n_obs, n_sectors), np.nan)
updated_state_variances = np.full((n_obs, n_sectors), np.nan)
parameter_rows = []

for i, sector in enumerate(sector_columns):
    initialization_residuals = residuals[:calibration_period, i]
    initialization_variance = np.var(initialization_residuals, ddof=1)
    assert initialization_variance > 0

    filtered_state = float(initialization_residuals.mean())
    filtered_variance = float(initialization_variance)

    autoregressive_path = []
    process_variance_path = []
    measurement_variance_path = []
    rolling_residual_variance_path = []

    for t in range(calibration_period, n_obs):
        # Re-estimate state-space parameters from a trailing window that ends at t-1.
        # This keeps the filter strictly causal and lets parameters adapt across regimes.
        window_end = t
        window_start = max(0, window_end - rolling_parameter_window)
        rolling_residual_window = residuals[window_start:window_end, i]
        assert len(rolling_residual_window) >= 3

        lagged_observations = rolling_residual_window[:-1]
        current_observations = rolling_residual_window[1:]
        design_matrix = np.column_stack((np.ones(len(lagged_observations)), lagged_observations))
        intercept_estimate, autoregressive_coefficient = np.linalg.lstsq(
            design_matrix,
            current_observations,
            rcond=None,
        )[0]
        autoregressive_coefficient = float(
            np.clip(autoregressive_coefficient, -0.95, 0.95)
        )

        innovations = current_observations - (
            intercept_estimate
            + autoregressive_coefficient * lagged_observations
        )
        innovation_variance = max(float(np.var(innovations, ddof=1)), 1e-10)
        rolling_residual_variance = max(
            float(np.var(rolling_residual_window, ddof=1)),
            1e-10,
        )

        # Use innovation variance as the basis for Q, and estimate R with a bounded
        # rolling-variance split to avoid unstable covariance inversions.
        innovation_share = innovation_variance / (
            innovation_variance + rolling_residual_variance
        )
        innovation_share = float(np.clip(innovation_share, 0.10, 0.90))
        process_variance = max(
            innovation_share * rolling_residual_variance,
            1e-10,
        )
        measurement_variance = max(
            (1.0 - innovation_share) * rolling_residual_variance,
            1e-10,
        )
        assert process_variance > 0
        assert measurement_variance > 0

        predicted_state = (
            intercept_estimate
            + autoregressive_coefficient * filtered_state
        )
        predicted_variance = (
            autoregressive_coefficient ** 2 * filtered_variance
            + process_variance
        )
        kalman_gain = predicted_variance / (
            predicted_variance + measurement_variance
        )
        updated_state = predicted_state + kalman_gain * (
            residuals[t, i] - predicted_state
        )
        updated_variance = (1 - kalman_gain) * predicted_variance
        assert updated_variance > 0

        state_estimates[t, i] = updated_state
        updated_state_variances[t, i] = updated_variance
        standardized_states[t, i] = updated_state / np.sqrt(
            updated_variance + measurement_variance
        )
        assert np.isfinite(standardized_states[t, i])
        state_changes[t, i] = updated_state - filtered_state

        filtered_state = updated_state
        filtered_variance = updated_variance

        autoregressive_path.append(float(autoregressive_coefficient))
        process_variance_path.append(float(process_variance))
        measurement_variance_path.append(float(measurement_variance))
        rolling_residual_variance_path.append(float(rolling_residual_variance))

    parameter_rows.append(
        {
            "Sector": sector,
            "AutoregressiveCoefficient": float(np.mean(autoregressive_path)),
            "ProcessVariance": float(np.mean(process_variance_path)),
            "MeasurementVariance": float(np.mean(measurement_variance_path)),
            "CalibrationResidualVariance": float(np.mean(rolling_residual_variance_path)),
        }
    )

valid_rows = np.isfinite(state_estimates).all(axis=1)
assert valid_rows.sum() == 86
assert (updated_state_variances[valid_rows] > 0).all()
assert np.isfinite(standardized_states[valid_rows]).all()
assert not np.isinf(standardized_states[valid_rows]).any()
assert np.mean(np.abs(standardized_states[valid_rows])) < 25.0

dynamic_mispricing_columns = [f"DynamicMispricing_{sector}" for sector in sector_columns]
standardized_mispricing_columns = [
    f"StandardizedMispricing_{sector}" for sector in sector_columns
]
relative_mispricing_columns = [
    f"RelativeMispricing_{sector}" for sector in sector_columns
]
mispricing_change_columns = [
    f"MispricingChange_{sector}" for sector in sector_columns
]

relative_mispricing = state_estimates[valid_rows] - state_estimates[valid_rows].mean(
    axis=1,
    keepdims=True,
)
valid_state_changes = state_changes[valid_rows]
valid_state_estimates = state_estimates[valid_rows]

D3 = pd.concat(
    [
        D2.loc[valid_rows, ["Date"]].reset_index(drop=True),
        pd.DataFrame(valid_state_estimates, columns=dynamic_mispricing_columns),
        pd.DataFrame(
            standardized_states[valid_rows],
            columns=standardized_mispricing_columns,
        ),
        pd.DataFrame(relative_mispricing, columns=relative_mispricing_columns),
        pd.DataFrame(valid_state_changes, columns=mispricing_change_columns),
        pd.DataFrame(
            {
                "CrossSectionalMispricingDispersion": np.std(
                    valid_state_estimates,
                    axis=1,
                    ddof=1,
                ),
                "SystemicMispricingMagnitude": np.sqrt(
                    np.mean(valid_state_estimates ** 2, axis=1)
                ),
            }
        ),
    ],
    axis=1,
)

filter_parameters = pd.DataFrame(parameter_rows)

assert D3.shape == (86, 39)
assert D3["Date"].duplicated().sum() == 0
assert D3.duplicated().sum() == 0
assert D3.isna().sum().sum() == 0
assert not np.isinf(D3.drop(columns="Date").to_numpy(dtype=float)).any()
assert D3["Date"].is_monotonic_increasing
assert D3["Date"].reset_index(drop=True).equals(
    pd.Series(pd.date_range(D3["Date"].min(), D3["Date"].max(), freq="ME"))
)
assert np.allclose(
    D3[relative_mispricing_columns].to_numpy().mean(axis=1),
    0.0,
    atol=1e-12,
)
assert (filter_parameters["ProcessVariance"] > 0).all()
assert (filter_parameters["MeasurementVariance"] > 0).all()
assert np.isfinite(filter_parameters.drop(columns="Sector").to_numpy()).all()

D3.to_csv("D3.csv", index=False)
filter_parameters.to_csv("mispricing_filter_parameters.csv", index=False)

mean_absolute_standardized_mispricing = float(
    np.abs(D3[standardized_mispricing_columns].to_numpy()).mean()
)
maximum_systemic_mispricing_index = int(D3["SystemicMispricingMagnitude"].idxmax())

module_3_validation = {
    "D3_shape": D3.shape,
    "date_range": f"{D3['Date'].min().date()} to {D3['Date'].max().date()}",
    "duplicate_dates": int(D3["Date"].duplicated().sum()),
    "duplicate_rows": int(D3.duplicated().sum()),
    "missing_values": int(D3.isna().sum().sum()),
    "infinite_values": int(np.isinf(D3.drop(columns="Date").to_numpy(dtype=float)).sum()),
    "monthly_continuity": True,
    "chronological_order": True,
    "relative_mispricing_centered": True,
    "positive_filter_variances": True,
    "mean_absolute_standardized_mispricing": mean_absolute_standardized_mispricing,
    "peak_systemic_mispricing_date": str(D3.loc[maximum_systemic_mispricing_index, "Date"].date()),
    "look_ahead_bias": "Filter parameters use only the trailing history available before each update; each state is updated one-sided with the current residual only.",
    "data_leakage": "No backward smoothing, centred rolling window, or future residual is used.",
}
module_3_validation



{'D3_shape': (86, 39),
 'date_range': '2019-01-31 to 2026-02-28',
 'duplicate_dates': 0,
 'duplicate_rows': 0,
 'missing_values': 0,
 'infinite_values': 0,
 'monthly_continuity': True,
 'chronological_order': True,
 'relative_mispricing_centered': True,
 'positive_filter_variances': True,
 'mean_absolute_standardized_mispricing': 0.46930426555713955,
 'peak_systemic_mispricing_date': '2020-03-31',
 'look_ahead_bias': 'Filter parameters use only the trailing history available before each update; each state is updated one-sided with the current residual only.',
 'data_leakage': 'No backward smoothing, centred rolling window, or future residual is used.'}

# Module 4

In [14]:
from scipy.stats import spearmanr
from sklearn.cluster import KMeans

D3 = pd.read_csv("D3.csv")
D3["Date"] = pd.to_datetime(D3["Date"])

assert D3.shape == (86, 39)
assert D3["Date"].duplicated().sum() == 0
assert D3.duplicated().sum() == 0
assert D3.isna().sum().sum() == 0
assert not np.isinf(D3.drop(columns="Date").to_numpy(dtype=float)).any()
assert D3["Date"].is_monotonic_increasing
assert D3["Date"].reset_index(drop=True).equals(
    pd.Series(pd.date_range(D3["Date"].min(), D3["Date"].max(), freq="ME"))
)

sector_columns = [
    "Materials", "Energy", "Financials", "Industrials", "Technology",
    "Consumer_Staples", "Utilities", "Healthcare", "Consumer_Discretionary",
]
mispricing_columns = [f"DynamicMispricing_{sector}" for sector in sector_columns]
mispricing_states = D3[mispricing_columns].to_numpy(dtype=float)

correlation_window = 24
n_obs, n_sectors = mispricing_states.shape
assert n_obs > correlation_window

def spectral_entropy(eigenvalues, tolerance=1e-12):
    positive_values = np.asarray(eigenvalues, dtype=float)
    positive_values = positive_values[positive_values > tolerance]
    probabilities = positive_values / positive_values.sum()
    return float(-(probabilities * np.log(probabilities)).sum())

feature_rows = []
diagnostic_rows = []

for t in range(correlation_window - 1, n_obs):
    history = mispricing_states[t - correlation_window + 1:t + 1]
    # Spearman correlation captures monotonic dependence, not only linear dependence.
    # Rank-based dependence is typically more robust to outliers and heavy-tailed
    # financial observations while leaving the rest of the graph pipeline unchanged.
    spearman_correlation = spearmanr(history, axis=0).correlation
    assert np.isfinite(spearman_correlation).all()

    full_adjacency = np.abs(spearman_correlation)
    np.fill_diagonal(full_adjacency, 0.0)
    full_adjacency = np.clip((full_adjacency + full_adjacency.T) / 2, 0.0, 1.0)

    # Sparse graphs are more informative for spectral analysis because they suppress
    # weak ubiquitous links and emphasize the dominant local dependency structure.
    n_neighbors = min(3, n_sectors - 1)
    sparse_adjacency = np.zeros_like(full_adjacency)
    for i in range(n_sectors):
        ranked_neighbors = np.argsort(full_adjacency[i])[::-1]
        ranked_neighbors = ranked_neighbors[ranked_neighbors != i]
        selected_neighbors = ranked_neighbors[:n_neighbors]
        sparse_adjacency[i, selected_neighbors] = full_adjacency[i, selected_neighbors]


    # k-nearest-neighbor sparsification preserves the strongest market relationships
    # while keeping original correlation magnitudes on retained edges.
    adjacency = np.clip((sparse_adjacency + sparse_adjacency.T) / 2, 0.0, 1.0)

    degree = adjacency.sum(axis=1)
    assert np.all(degree > 0)
    degree_matrix = np.diag(degree)
    inverse_sqrt_degree = np.diag(1 / np.sqrt(degree))
    inverse_degree = np.diag(1 / degree)

    laplacian = degree_matrix - adjacency
    normalized_laplacian = (
        np.eye(n_sectors)
        - inverse_sqrt_degree @ adjacency @ inverse_sqrt_degree
    )
    random_walk_laplacian = np.eye(n_sectors) - inverse_degree @ adjacency

    laplacian_pseudoinverse = np.linalg.pinv(laplacian, hermitian=True)
    effective_resistance = (
        np.diag(laplacian_pseudoinverse)[:, None]
        + np.diag(laplacian_pseudoinverse)[None, :]
        - 2 * laplacian_pseudoinverse
    )
    effective_resistance = np.maximum(
        (effective_resistance + effective_resistance.T) / 2,
        0.0,
    )
    resistance_adjacency = np.zeros_like(effective_resistance)
    off_diagonal = ~np.eye(n_sectors, dtype=bool)
    resistance_adjacency[off_diagonal] = 1 / np.maximum(
        effective_resistance[off_diagonal],
        1e-10,
    )
    resistance_degree = resistance_adjacency.sum(axis=1)
    resistance_laplacian = (
        np.diag(resistance_degree) - resistance_adjacency
    )

    laplacian_eigenvalues = np.linalg.eigvalsh(laplacian)
    normalized_eigenvalues, normalized_eigenvectors = np.linalg.eigh(
        normalized_laplacian
    )
    random_walk_eigenvalues = np.linalg.eigvals(random_walk_laplacian)
    resistance_eigenvalues = np.linalg.eigvalsh(resistance_laplacian)

    assert np.max(np.abs(random_walk_eigenvalues.imag)) < 1e-10
    random_walk_eigenvalues = np.sort(random_walk_eigenvalues.real)

    leading_adjacency_eigenvalues, leading_adjacency_eigenvectors = np.linalg.eigh(
        adjacency
    )
    eigenvector_centrality = np.abs(leading_adjacency_eigenvectors[:, -1])
    eigenvector_centrality /= eigenvector_centrality.sum()

    # Spectral clustering with KMeans on the non-trivial normalized-Laplacian embedding
    # is the standard partitioning approach and is more robust than median-thresholding.
    spectral_embedding = normalized_eigenvectors[:, 1:3]
    community_labels = KMeans(
        n_clusters=2,
        n_init=20,
        random_state=0,
    ).fit_predict(spectral_embedding)

    total_edge_weight = adjacency.sum()
    same_community = community_labels[:, None] == community_labels[None, :]
    modularity = float(
        (
            adjacency
            - np.outer(degree, degree) / total_edge_weight
        )[same_community].sum() / total_edge_weight
    )

    row = {
        "Date": D3.loc[t, "Date"],
        "MeanAbsoluteCorrelation": float(adjacency[off_diagonal].mean()),
        "AlgebraicConnectivity": float(laplacian_eigenvalues[1]),
        "LaplacianSpectralRadius": float(laplacian_eigenvalues[-1]),
        "NormalizedAlgebraicConnectivity": float(normalized_eigenvalues[1]),
        "NormalizedSpectralEntropy": spectral_entropy(normalized_eigenvalues),
        "RandomWalkSpectralGap": float(random_walk_eigenvalues[1]),
        "ResistanceAlgebraicConnectivity": float(resistance_eigenvalues[1]),
        "ResistanceSpectralEntropy": spectral_entropy(resistance_eigenvalues),
        "SpectralPartitionModularity": modularity,
    }
    row.update(
        {
            f"WeightedDegree_{sector}": float(degree[i])
            for i, sector in enumerate(sector_columns)
        }
    )
    row.update(
        {
            f"EigenvectorCentrality_{sector}": float(eigenvector_centrality[i])
            for i, sector in enumerate(sector_columns)
        }
    )
    row.update(
        {
            f"SpectralCommunity_{sector}": int(community_labels[i])
            for i, sector in enumerate(sector_columns)
        }
    )
    feature_rows.append(row)

    diagnostic_rows.append(
        {
            "Date": D3.loc[t, "Date"],
            "LaplacianSymmetric": bool(np.allclose(laplacian, laplacian.T)),
            "LaplacianZeroRowSum": bool(np.allclose(laplacian.sum(axis=1), 0.0)),
            "NormalizedLaplacianSymmetric": bool(
                np.allclose(normalized_laplacian, normalized_laplacian.T)
            ),
            "RandomWalkLaplacianZeroRowSum": bool(
                np.allclose(random_walk_laplacian.sum(axis=1), 0.0)
            ),
            "ResistanceLaplacianSymmetric": bool(
                np.allclose(resistance_laplacian, resistance_laplacian.T)
            ),
            "ResistanceLaplacianZeroRowSum": bool(
                np.allclose(resistance_laplacian.sum(axis=1), 0.0)
            ),
            "NonnegativeLaplacianSpectrum": bool(
                np.all(laplacian_eigenvalues >= -1e-10)
            ),
            "NormalizedSpectrumInRange": bool(
                np.all(normalized_eigenvalues >= -1e-10)
                and np.all(normalized_eigenvalues <= 2 + 1e-10)
            ),
            "FiniteEffectiveResistance": bool(np.isfinite(effective_resistance).all()),
        }
    )

D4 = pd.DataFrame(feature_rows)
graph_operator_diagnostics = pd.DataFrame(diagnostic_rows)

assert D4.shape == (63, 37)
assert D4["Date"].duplicated().sum() == 0
assert D4.duplicated().sum() == 0
assert D4.isna().sum().sum() == 0
assert not np.isinf(D4.drop(columns="Date").to_numpy(dtype=float)).any()
assert D4["Date"].is_monotonic_increasing
assert D4["Date"].reset_index(drop=True).equals(
    pd.Series(pd.date_range(D4["Date"].min(), D4["Date"].max(), freq="ME"))
)
assert graph_operator_diagnostics.shape == (63, 10)
assert graph_operator_diagnostics.drop(columns="Date").to_numpy(dtype=bool).all()
assert (D4["WeightedDegree_Materials"] > 0).all()
assert np.allclose(
    D4[[f"EigenvectorCentrality_{sector}" for sector in sector_columns]].sum(axis=1),
    1.0,
 )
assert D4[[f"SpectralCommunity_{sector}" for sector in sector_columns]].isin([0, 1]).all().all()

D4.to_csv("D4.csv", index=False)
graph_operator_diagnostics.to_csv("graph_operator_diagnostics.csv", index=False)

module_4_validation = {
    "D4_shape": D4.shape,
    "date_range": f"{D4['Date'].min().date()} to {D4['Date'].max().date()}",
    "duplicate_dates": int(D4["Date"].duplicated().sum()),
    "duplicate_rows": int(D4.duplicated().sum()),
    "missing_values": int(D4.isna().sum().sum()),
    "infinite_values": int(np.isinf(D4.drop(columns="Date").to_numpy(dtype=float)).sum()),
    "monthly_continuity": True,
    "chronological_order": True,
    "all_operator_diagnostics_pass": True,
    "mean_absolute_correlation": float(D4["MeanAbsoluteCorrelation"].mean()),
    "mean_normalized_spectral_entropy": float(D4["NormalizedSpectralEntropy"].mean()),
    "look_ahead_bias": "Each graph uses a trailing 24-month window ending at its own date.",
    "data_leakage": "No future mispricing state, correlation, graph weight, or spectral decomposition is used.",
}
module_4_validation



{'D4_shape': (63, 37),
 'date_range': '2020-12-31 to 2026-02-28',
 'duplicate_dates': 0,
 'duplicate_rows': 0,
 'missing_values': 0,
 'infinite_values': 0,
 'monthly_continuity': True,
 'chronological_order': True,
 'all_operator_diagnostics_pass': True,
 'mean_absolute_correlation': 0.28280499961659383,
 'mean_normalized_spectral_entropy': 2.022203140770718,
 'look_ahead_bias': 'Each graph uses a trailing 24-month window ending at its own date.',
 'data_leakage': 'No future mispricing state, correlation, graph weight, or spectral decomposition is used.'}

# Module 5

In [16]:
from ripser import ripser

D3 = pd.read_csv("D3.csv")
D4 = pd.read_csv("D4.csv")
D3["Date"] = pd.to_datetime(D3["Date"])

D4["Date"] = pd.to_datetime(D4["Date"])

assert D3["Date"].duplicated().sum() == 0
assert D4["Date"].duplicated().sum() == 0
assert D3["Date"].is_monotonic_increasing
assert D4["Date"].is_monotonic_increasing
assert D4["Date"].isin(D3["Date"]).all()

sector_columns = [
    "Materials", "Energy", "Financials", "Industrials", "Technology",
    "Consumer_Staples", "Utilities", "Healthcare", "Consumer_Discretionary",
]
standardized_mispricing_columns = [
    f"StandardizedMispricing_{sector}" for sector in sector_columns
]
mispricing_states = D3[standardized_mispricing_columns].to_numpy(dtype=float)
date_to_index = {date: index for index, date in enumerate(D3["Date"])}

trajectory_window = 12
reported_filtration_grid = np.array([0.5, 1.0, 1.5, 2.0])
dense_grid_points = 21

def finite_persistence(diagram):
    finite_bars = diagram[np.isfinite(diagram[:, 1])]
    return finite_bars[:, 1] - finite_bars[:, 0]

def persistence_entropy(persistence):
    positive_persistence = persistence[persistence > 1e-12]
    if len(positive_persistence) == 0:
        return 0.0
    probabilities = positive_persistence / positive_persistence.sum()
    return float(-(probabilities * np.log(probabilities)).sum())

def betti_number(diagram, epsilon):
    return int(np.sum((diagram[:, 0] <= epsilon) & (epsilon < diagram[:, 1])))

def laplacian_entropy(eigenvalues):
    positive_values = eigenvalues[eigenvalues > 1e-12]
    if len(positive_values) == 0:
        return 0.0
    probabilities = positive_values / positive_values.sum()
    return float(-(probabilities * np.log(probabilities)).sum())

feature_rows = []
diagram_rows = []

for date in D4["Date"]:
    t = date_to_index[date]
    assert t >= trajectory_window - 1

    trajectory = mispricing_states[t - trajectory_window + 1:t + 1].T
    trajectory_mean = trajectory.mean(axis=0)
    trajectory_scale = trajectory.std(axis=0, ddof=1)
    assert np.all(trajectory_scale > 0)
    point_cloud = (trajectory - trajectory_mean) / trajectory_scale

    pairwise_distances = np.sqrt(
        ((point_cloud[:, None, :] - point_cloud[None, :, :]) ** 2).sum(axis=2)
    )
    positive_distances = pairwise_distances[
        np.triu_indices(len(sector_columns), k=1)
    ]
    median_distance = float(np.median(positive_distances))
    assert median_distance > 0
    normalized_distances = pairwise_distances / median_distance
    positive_normalized_distances = normalized_distances[
        np.triu_indices(len(sector_columns), k=1)
    ]

    persistence_result = ripser(
        normalized_distances,
        distance_matrix=True,
        maxdim=1,
    )
    diagram_h0, diagram_h1 = persistence_result["dgms"]
    assert np.isfinite(diagram_h0[:, 0]).all()
    assert np.isfinite(diagram_h1).all()

    h0_persistence = finite_persistence(diagram_h0)
    h1_persistence = finite_persistence(diagram_h1)

    # The filtration threshold is adaptive and uses only the current window geometry,
    # making the threshold graph responsive to the present cross-sector distance regime.
    adaptive_filtration_threshold = float(np.quantile(positive_normalized_distances, 0.75))
    threshold_graph_adjacency = (
        (normalized_distances <= adaptive_filtration_threshold)
        & (~np.eye(len(sector_columns), dtype=bool))
    ).astype(float)
    threshold_graph_degree = threshold_graph_adjacency.sum(axis=1)
    threshold_graph_laplacian = np.diag(threshold_graph_degree) - threshold_graph_adjacency
    threshold_graph_eigenvalues = np.linalg.eigvalsh(threshold_graph_laplacian)

    # A denser filtration grid gives a more faithful numerical approximation of Betti
    # curves while preserving the existing reported Betti summary columns.
    dense_grid_upper = max(
        2.0,
        float(np.quantile(positive_normalized_distances, 0.95)),
    )
    dense_filtration_grid = np.linspace(0.0, dense_grid_upper, dense_grid_points)

    row = {
        "Date": date,
        "H0TotalPersistence": float(h0_persistence.sum()),
        "H0PersistentEntropy": persistence_entropy(h0_persistence),
        "H1FeatureCount": int(len(h1_persistence)),
        "H1TotalPersistence": float(h1_persistence.sum()),
        "H1PersistentEntropy": persistence_entropy(h1_persistence),
        "H1MaxPersistence": float(h1_persistence.max()) if len(h1_persistence) else 0.0,
        "Betti0Area": float(np.trapz(
            [betti_number(diagram_h0, epsilon) for epsilon in dense_filtration_grid],
            dense_filtration_grid,
        )),
        "Betti1Area": float(np.trapz(
            [betti_number(diagram_h1, epsilon) for epsilon in dense_filtration_grid],
            dense_filtration_grid,
        )),
        # This is intentionally named as a threshold-graph Laplacian to distinguish it
        # from the persistent Laplacian object used in TDA literature.
        "ThresholdGraphLaplacianSpectralGap": float(
            threshold_graph_eigenvalues[1]
        ),
        "ThresholdGraphLaplacianEntropy": laplacian_entropy(threshold_graph_eigenvalues),
        "ThresholdGraphLaplacianZeroEigenvalues": int(
            np.sum(threshold_graph_eigenvalues < 1e-10)
        ),
        "ThresholdGraphLaplacianTrace": float(np.trace(threshold_graph_laplacian)),
        "PointCloudMedianDistance": median_distance,
        "PointCloudDiameter": float(normalized_distances.max()),
    }
    row.update(
        {
            f"Betti0_{epsilon:.1f}": betti_number(diagram_h0, epsilon)
            for epsilon in reported_filtration_grid
        }
    )
    row.update(
        {
            f"Betti1_{epsilon:.1f}": betti_number(diagram_h1, epsilon)
            for epsilon in reported_filtration_grid
        }
    )
    feature_rows.append(row)

    for dimension, diagram in enumerate((diagram_h0, diagram_h1)):
        for birth, death in diagram:
            if np.isfinite(death):
                diagram_rows.append(
                    {
                        "Date": date,
                        "Dimension": dimension,
                        "Birth": float(birth),
                        "Death": float(death),
                        "Persistence": float(death - birth),
                    }
                )

D5 = pd.DataFrame(feature_rows)
persistence_diagrams = pd.DataFrame(diagram_rows)

assert D5.shape == (63, 23)
assert D5["Date"].duplicated().sum() == 0
assert D5.duplicated().sum() == 0
assert D5.isna().sum().sum() == 0
assert not np.isinf(D5.drop(columns="Date").to_numpy(dtype=float)).any()
assert D5["Date"].is_monotonic_increasing
assert D5["Date"].reset_index(drop=True).equals(
    pd.Series(pd.date_range(D5["Date"].min(), D5["Date"].max(), freq="ME"))
)
assert set(persistence_diagrams["Dimension"].unique()).issubset({0, 1})
assert persistence_diagrams["Persistence"].ge(0).all()
assert np.isfinite(persistence_diagrams.drop(columns="Date").to_numpy(dtype=float)).all()
assert D5["ThresholdGraphLaplacianZeroEigenvalues"].ge(1).all()
assert D5["PointCloudMedianDistance"].gt(0).all()

D5.to_csv("D5.csv", index=False)
persistence_diagrams.to_csv("persistence_diagrams.csv", index=False)

module_5_validation = {
    "D5_shape": D5.shape,
    "date_range": f"{D5['Date'].min().date()} to {D5['Date'].max().date()}",
    "duplicate_dates": int(D5["Date"].duplicated().sum()),
    "duplicate_rows": int(D5.duplicated().sum()),
    "missing_values": int(D5.isna().sum().sum()),
    "infinite_values": int(np.isinf(D5.drop(columns="Date").to_numpy(dtype=float)).sum()),
    "monthly_continuity": True,
    "chronological_order": True,
    "persistence_diagram_rows": int(len(persistence_diagrams)),
    "mean_h1_total_persistence": float(D5["H1TotalPersistence"].mean()),
    "mean_betti1_area": float(D5["Betti1Area"].mean()),
    "look_ahead_bias": "Each point cloud uses only the 12-month trajectory ending at its own date.",
    "data_leakage": "No future mispricing state, distance, filtration, persistence interval, or Laplacian is used.",
}
module_5_validation



{'D5_shape': (63, 23),
 'date_range': '2020-12-31 to 2026-02-28',
 'duplicate_dates': 0,
 'duplicate_rows': 0,
 'missing_values': 0,
 'infinite_values': 0,
 'monthly_continuity': True,
 'chronological_order': True,
 'persistence_diagram_rows': 534,
 'mean_h1_total_persistence': 0.021779807787092906,
 'mean_betti1_area': 0.028571428571428574,
 'look_ahead_bias': 'Each point cloud uses only the 12-month trajectory ending at its own date.',
 'data_leakage': 'No future mispricing state, distance, filtration, persistence interval, or Laplacian is used.'}

# Module 6

In [20]:
from scipy.optimize import linprog

D3 = pd.read_csv("D3.csv")
D5 = pd.read_csv("D5.csv")
D3["Date"] = pd.to_datetime(D3["Date"])
D5["Date"] = pd.to_datetime(D5["Date"])

assert D3["Date"].duplicated().sum() == 0
assert D5["Date"].duplicated().sum() == 0
assert D5["Date"].isin(D3["Date"]).all()
assert D5["Date"].is_monotonic_increasing

sector_columns = [
    "Materials", "Energy", "Financials", "Industrials", "Technology",
    "Consumer_Staples", "Utilities", "Healthcare", "Consumer_Discretionary",
]
standardized_mispricing_columns = [
    f"StandardizedMispricing_{sector}" for sector in sector_columns
]
dynamic_mispricing_columns = [
    f"DynamicMispricing_{sector}" for sector in sector_columns
]
mispricing_states = D3[standardized_mispricing_columns].to_numpy(dtype=float)
dynamic_mispricing_states = D3[dynamic_mispricing_columns].to_numpy(dtype=float)
date_to_index = {date: index for index, date in enumerate(D3["Date"])}

trajectory_window = 12
n_sectors = len(sector_columns)

transport_constraint = np.zeros((2 * n_sectors, n_sectors * n_sectors))
for source in range(n_sectors):
    transport_constraint[source, source * n_sectors:(source + 1) * n_sectors] = 1
for target in range(n_sectors):
    transport_constraint[n_sectors + target, target::n_sectors] = 1

transport_rows = []
plan_rows = []
marginal_rows = []

for current_date, previous_date in zip(D5["Date"].iloc[1:], D5["Date"].iloc[:-1]):
    current_index = date_to_index[current_date]
    previous_index = date_to_index[previous_date]
    assert current_index == previous_index + 1
    assert previous_index >= trajectory_window - 1

    previous_trajectory = mispricing_states[
        previous_index - trajectory_window + 1:previous_index + 1
    ].T
    current_trajectory = mispricing_states[
        current_index - trajectory_window + 1:current_index + 1
    ].T

    pooled_trajectory = np.vstack((previous_trajectory, current_trajectory))
    pooled_mean = pooled_trajectory.mean(axis=0)
    pooled_scale = pooled_trajectory.std(axis=0, ddof=1)
    assert np.all(pooled_scale > 0)

    previous_cloud = (previous_trajectory - pooled_mean) / pooled_scale
    current_cloud = (current_trajectory - pooled_mean) / pooled_scale
    ground_cost = np.sqrt(
        ((previous_cloud[:, None, :] - current_cloud[None, :, :]) ** 2).sum(axis=2)
    )

    # Transport masses are proportional to the absolute latent dynamic mispricing
    # (Kalman-filter state), so mass tracks economically meaningful disequilibrium
    # magnitude rather than standardized z-score units.
    source_mass_raw = np.abs(dynamic_mispricing_states[previous_index])
    target_mass_raw = np.abs(dynamic_mispricing_states[current_index])
    assert source_mass_raw.sum() > 0
    assert target_mass_raw.sum() > 0

    # Normalization preserves balanced optimal transport: both marginals sum to one,
    # so the plan still transports a single unit of probability mass.
    source_mass = source_mass_raw / source_mass_raw.sum()
    target_mass = target_mass_raw / target_mass_raw.sum()
    assert np.isclose(source_mass.sum(), 1.0)
    assert np.isclose(target_mass.sum(), 1.0)

    transport_constraint_rhs = np.concatenate((source_mass, target_mass))

    solution = linprog(
        ground_cost.reshape(-1),
        A_eq=transport_constraint,
        b_eq=transport_constraint_rhs,
        bounds=(0, None),
        method="highs",
    )
    assert solution.success, solution.message

    transport_plan = np.maximum(solution.x.reshape(n_sectors, n_sectors), 0.0)
    transport_plan /= transport_plan.sum()
    assert np.allclose(transport_plan.sum(axis=1), source_mass, atol=1e-8)
    assert np.allclose(transport_plan.sum(axis=0), target_mass, atol=1e-8)

    earth_movers_distance = float((transport_plan * ground_cost).sum())
    diagonal_mass = float(np.trace(transport_plan))
    fixed_identity_displacement = float(
        np.sqrt(((current_cloud - previous_cloud) ** 2).sum(axis=1)).mean()
    )

    transport_rows.append(
        {
            "Date": current_date,
            "TrajectoryEarthMoversDistance": earth_movers_distance,
            "DiagonalTransportMass": diagonal_mass,
            "ReallocatedTransportMass": float(1 - diagonal_mass),
            "FixedIdentityTrajectoryDisplacement": fixed_identity_displacement,
            "MaximumTransportCost": float(ground_cost.max()),
            "TransportPlanEntropy": float(
                -np.sum(
                    transport_plan[transport_plan > 0]
                    * np.log(transport_plan[transport_plan > 0])
                )
            ),
        }
    )

    for sector_index, sector in enumerate(sector_columns):
        marginal_rows.append(
            {
                "Date": current_date,
                "Sector": sector,
                "SourceMass": float(source_mass[sector_index]),
                "TargetMass": float(target_mass[sector_index]),
            }
        )

    for source_index, source_sector in enumerate(sector_columns):
        for target_index, target_sector in enumerate(sector_columns):
            plan_rows.append(
                {
                    "Date": current_date,
                    "SourceSector": source_sector,
                    "TargetSector": target_sector,
                    "TransportMass": float(transport_plan[source_index, target_index]),
                    "GroundCost": float(ground_cost[source_index, target_index]),
                }
            )

D6 = pd.DataFrame(transport_rows)
optimal_transport_plans = pd.DataFrame(plan_rows)
adaptive_marginals = pd.DataFrame(marginal_rows)

assert D6.shape == (62, 7)
assert D6["Date"].duplicated().sum() == 0
assert D6.duplicated().sum() == 0
assert D6.isna().sum().sum() == 0
assert not np.isinf(D6.drop(columns="Date").to_numpy(dtype=float)).any()
assert D6["Date"].is_monotonic_increasing
assert D6["Date"].reset_index(drop=True).equals(
    pd.Series(pd.date_range(D6["Date"].min(), D6["Date"].max(), freq="ME"))
)
assert optimal_transport_plans.shape == (62 * n_sectors * n_sectors, 5)
assert optimal_transport_plans.isna().sum().sum() == 0
assert optimal_transport_plans["TransportMass"].ge(0).all()
assert np.isfinite(optimal_transport_plans[["TransportMass", "GroundCost"]].to_numpy()).all()

assert adaptive_marginals.shape == (62 * n_sectors, 4)
source_plan_masses = (
    optimal_transport_plans.groupby(["Date", "SourceSector"])
    ["TransportMass"]
    .sum()
    .reset_index()
    .rename(columns={"SourceSector": "Sector", "TransportMass": "PlanSourceMass"})
)
target_plan_masses = (
    optimal_transport_plans.groupby(["Date", "TargetSector"])
    ["TransportMass"]
    .sum()
    .reset_index()
    .rename(columns={"TargetSector": "Sector", "TransportMass": "PlanTargetMass"})
)
mass_consistency = adaptive_marginals.merge(
    source_plan_masses,
    on=["Date", "Sector"],
    how="inner",
    validate="one_to_one",
).merge(
    target_plan_masses,
    on=["Date", "Sector"],
    how="inner",
    validate="one_to_one",
)
assert mass_consistency.shape == (62 * n_sectors, 6)
assert np.allclose(
    mass_consistency["PlanSourceMass"].to_numpy(),
    mass_consistency["SourceMass"].to_numpy(),
    atol=1e-8,
)
assert np.allclose(
    mass_consistency["PlanTargetMass"].to_numpy(),
    mass_consistency["TargetMass"].to_numpy(),
    atol=1e-8,
)

D6.to_csv("D6.csv", index=False)
optimal_transport_plans.to_csv("optimal_transport_plans.csv", index=False)

module_6_validation = {
    "D6_shape": D6.shape,
    "date_range": f"{D6['Date'].min().date()} to {D6['Date'].max().date()}",
    "duplicate_dates": int(D6["Date"].duplicated().sum()),
    "duplicate_rows": int(D6.duplicated().sum()),
    "missing_values": int(D6.isna().sum().sum()),
    "infinite_values": int(np.isinf(D6.drop(columns="Date").to_numpy(dtype=float)).sum()),
    "monthly_continuity": True,
    "chronological_order": True,
    "balanced_transport_marginals": True,
    "mean_trajectory_earth_movers_distance": float(D6["TrajectoryEarthMoversDistance"].mean()),
    "look_ahead_bias": "Each transport comparison uses only the current and immediately preceding trailing trajectory windows.",
    "data_leakage": "No future mispricing state, point cloud, cost matrix, or transport plan is used.",
}
module_6_validation



{'D6_shape': (62, 7),
 'date_range': '2021-01-31 to 2026-02-28',
 'duplicate_dates': 0,
 'duplicate_rows': 0,
 'missing_values': 0,
 'infinite_values': 0,
 'monthly_continuity': True,
 'chronological_order': True,
 'balanced_transport_marginals': True,
 'mean_trajectory_earth_movers_distance': 5.158018374426853,
 'look_ahead_bias': 'Each transport comparison uses only the current and immediately preceding trailing trajectory windows.',
 'data_leakage': 'No future mispricing state, point cloud, cost matrix, or transport plan is used.'}

# Module 7

In [27]:
D1 = pd.read_csv("D1.csv")
D2 = pd.read_csv("D2.csv")
D3 = pd.read_csv("D3.csv")
D4 = pd.read_csv("D4.csv")
D5 = pd.read_csv("D5.csv")
D6 = pd.read_csv("D6.csv")

datasets = {
    "D1": D1,
    "D2": D2,
    "D3": D3,
    "D4": D4,
    "D5": D5,
    "D6": D6,
}

for name, frame in datasets.items():
    frame["Date"] = pd.to_datetime(frame["Date"])
    assert frame["Date"].duplicated().sum() == 0, f"{name} has duplicate dates."
    assert frame.duplicated().sum() == 0, f"{name} has duplicate rows."
    assert frame.isna().sum().sum() == 0, f"{name} has missing values."
    assert not np.isinf(frame.drop(columns="Date").to_numpy(dtype=float)).any(), f"{name} has infinite values."
    assert frame["Date"].is_monotonic_increasing, f"{name} is not chronological."

sector_columns = [
    "Materials", "Energy", "Financials", "Industrials", "Technology",
    "Consumer_Staples", "Utilities", "Healthcare", "Consumer_Discretionary",
]
equilibrium_columns = [f"EquilibriumExcessReturn_{sector}" for sector in sector_columns]
residual_columns = [f"EquilibriumResidual_{sector}" for sector in sector_columns]
latent_columns = ["LatentFactor1", "LatentFactor2"]
selected_d2_columns = ["Date"] + equilibrium_columns + residual_columns + latent_columns

assert all(column in D2.columns for column in selected_d2_columns)
assert all(
    f"ActualExcessReturn_{sector}" not in selected_d2_columns
    for sector in sector_columns
)

source_frames = [
    ("D1", D1),
    ("D2", D2[selected_d2_columns]),
    ("D3", D3),
    ("D4", D4),
    ("D5", D5),
    ("D6", D6),
]
common_dates = set.intersection(*(set(frame["Date"]) for _, frame in source_frames))
assert len(common_dates) == len(D6)

merged = D6[["Date"]].copy()
for source_name, frame in source_frames:
    numeric_columns = [column for column in frame.columns if column != "Date"]
    assert len(numeric_columns) == len(set(numeric_columns))
    merged = merged.merge(
        frame[["Date"] + numeric_columns],
        on="Date",
        how="inner",
        validate="one_to_one",
    )

initial_feature_names = [column for column in merged.columns if column != "Date"]
assert len(initial_feature_names) == len(set(initial_feature_names))

# Correlation filtering removes redundant linear information while preserving interpretability
# because kept features remain original, named engineered variables rather than transformed factors.
correlation_threshold = 0.95
correlation_filter_rows = []

# This is unsupervised and target-free: only contemporaneous fused features are used,
# so no label information or future target leakage can enter this module.
absolute_correlation = (
    merged[initial_feature_names].corr(method="pearson").abs().fillna(0.0)
)
absolute_correlation_matrix = absolute_correlation.to_numpy()
removed_features = set()
n_features = len(initial_feature_names)

# Single deterministic upper-triangular scan: keep earlier features, remove later highly
# correlated ones, and never remove a feature more than once.
for i in range(n_features - 1):
    retained_feature = initial_feature_names[i]
    if retained_feature in removed_features:
        continue
    for j in range(i + 1, n_features):
        removed_feature = initial_feature_names[j]
        if removed_feature in removed_features:
            continue
        pair_correlation = float(absolute_correlation_matrix[i, j])
        if pair_correlation > correlation_threshold:
            removed_features.add(removed_feature)
            correlation_filter_rows.append(
                {
                    "RetainedFeature": retained_feature,
                    "RemovedFeature": removed_feature,
                    "AbsoluteCorrelation": pair_correlation,
                }
            )

retained_features = [
    feature for feature in initial_feature_names if feature not in removed_features
]
D7 = merged[["Date"] + retained_features].copy()
feature_names = retained_features.copy()
correlation_filter_manifest = pd.DataFrame(correlation_filter_rows)

assert len(feature_names) == len(set(feature_names))
assert D7.shape[0] == len(D6)
assert D7.shape[1] == len(feature_names) + 1
assert D7["Date"].duplicated().sum() == 0
assert D7.duplicated().sum() == 0
assert D7.isna().sum().sum() == 0
assert not np.isinf(D7.drop(columns="Date").to_numpy(dtype=float)).any()
assert D7["Date"].is_monotonic_increasing
assert D7["Date"].reset_index(drop=True).equals(
    pd.Series(pd.date_range(D7["Date"].min(), D7["Date"].max(), freq="ME"))
)

if len(feature_names) > 1:
    final_absolute_correlation = (
        D7[feature_names].corr(method="pearson").abs().fillna(0.0)
    )
    np.fill_diagonal(final_absolute_correlation.values, 0.0)
    assert final_absolute_correlation.to_numpy().max() <= correlation_threshold + 1e-12

assert set(row["RemovedFeature"] for row in correlation_filter_rows).isdisjoint(set(feature_names))
assert set(feature_names).issubset(set(initial_feature_names))
assert len(feature_names) + len(correlation_filter_rows) == len(initial_feature_names)
assert len(removed_features) == len(correlation_filter_manifest)
assert correlation_filter_manifest["RemovedFeature"].is_unique

correlation_filter_manifest = correlation_filter_manifest.sort_values(
    "AbsoluteCorrelation", ascending=False
).reset_index(drop=True)

manifest_rows = []
for source_name, frame in source_frames:
    allowed_columns = [
        column for column in frame.columns
        if column != "Date" and column in feature_names
    ]
    manifest_rows.extend(
        {"Feature": column, "Source": source_name}
        for column in allowed_columns
    )
feature_manifest = pd.DataFrame(manifest_rows)
assert feature_manifest.shape == (len(feature_names), 2)
assert feature_manifest["Feature"].is_unique
assert set(feature_manifest["Feature"]) == set(feature_names)

D7.to_csv("D7.csv", index=False)
feature_manifest.to_csv("feature_manifest.csv", index=False)
correlation_filter_manifest.to_csv("correlation_filter_manifest.csv", index=False)

source_feature_counts = feature_manifest.groupby("Source")["Feature"].count().to_dict()

# Scaling is intentionally deferred to the forecasting module, where transformations are
# fit only inside walk-forward training windows to avoid cross-period leakage.
module_7_validation = {
    "D7_shape": D7.shape,
    "date_range": f"{D7['Date'].min().date()} to {D7['Date'].max().date()}",
    "common_date_count": int(len(common_dates)),
    "duplicate_dates": int(D7["Date"].duplicated().sum()),
    "duplicate_rows": int(D7.duplicated().sum()),
    "missing_values": int(D7.isna().sum().sum()),
    "infinite_values": int(np.isinf(D7.drop(columns="Date").to_numpy(dtype=float)).sum()),
    "monthly_continuity": True,
    "chronological_order": True,
    "unique_feature_names": True,
    "source_feature_counts": source_feature_counts,
    "correlation_filter_threshold": correlation_threshold,
    "initial_feature_count": int(len(initial_feature_names)),
    "retained_feature_count": int(len(feature_names)),
    "removed_feature_count": int(len(correlation_filter_manifest)),
    "look_ahead_bias": "Fusion is an exact date intersection; it performs no temporal fill, interpolation, or future-aware transformation.",
    "data_leakage": "Correlation filtering is unsupervised and uses only fused features; scaling remains deferred to walk-forward modelling.",
}
module_7_validation



{'D7_shape': (62, 166),
 'date_range': '2021-01-31 to 2026-02-28',
 'common_date_count': 62,
 'duplicate_dates': 0,
 'duplicate_rows': 0,
 'missing_values': 0,
 'infinite_values': 0,
 'monthly_continuity': True,
 'chronological_order': True,
 'unique_feature_names': True,
 'source_feature_counts': {'D1': 65,
  'D2': 20,
  'D3': 22,
  'D4': 32,
  'D5': 21,
  'D6': 5},
 'correlation_filter_threshold': 0.95,
 'initial_feature_count': 207,
 'retained_feature_count': 165,
 'removed_feature_count': 42,
 'look_ahead_bias': 'Fusion is an exact date intersection; it performs no temporal fill, interpolation, or future-aware transformation.',
 'data_leakage': 'Correlation filtering is unsupervised and uses only fused features; scaling remains deferred to walk-forward modelling.'}

# Module 8

In [ ]:
from sklearn.feature_selection import SelectFromModel, VarianceThreshold
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
import warnings
import xgboost.core as xgb_core

warnings.filterwarnings("ignore", category=UserWarning)

# Work around a known XGBoost DataIter cleanup bug on some Python/XGBoost builds.
# This does not change training or prediction logic; it only prevents a spurious
# ignored exception during object finalization.
if hasattr(xgb_core, "DataIter") and not hasattr(xgb_core.DataIter, "_copilot_safe_del_patch"):
    _xgb_dataiter_original_del = getattr(xgb_core.DataIter, "__del__", None)

    def _xgb_dataiter_safe_del(self):
        if not hasattr(self, "_temporary_data"):
            self._temporary_data = None
        if _xgb_dataiter_original_del is not None:
            try:
                _xgb_dataiter_original_del(self)
            except AttributeError:
                pass

    xgb_core.DataIter.__del__ = _xgb_dataiter_safe_del
    xgb_core.DataIter._copilot_safe_del_patch = True

D0 = pd.read_csv("D0.csv")
D1 = pd.read_csv("D1.csv")
D7 = pd.read_csv("D7.csv")
for frame in (D0, D1, D7):
    frame["Date"] = pd.to_datetime(frame["Date"])

assert D7["Date"].duplicated().sum() == 0
assert D7.isna().sum().sum() == 0
assert not np.isinf(D7.drop(columns="Date").to_numpy(dtype=float)).any()
assert D7["Date"].is_monotonic_increasing

sector_columns = [
    "Materials", "Energy", "Financials", "Industrials", "Technology",
    "Consumer_Staples", "Utilities", "Healthcare", "Consumer_Discretionary",
]
excess_columns = [f"ExcessLogReturn_{sector}" for sector in sector_columns]
target_columns = [f"TargetExcessLogReturn_{sector}" for sector in sector_columns]

target_frame = D1[["Date"] + excess_columns].rename(
    columns=dict(zip(excess_columns, target_columns))
)
target_frame = target_frame.rename(columns={"Date": "TargetDate"})
forecasting_panel = D7.copy()
forecasting_panel["TargetDate"] = forecasting_panel["Date"] + pd.offsets.MonthEnd(1)
forecasting_panel = forecasting_panel.merge(
    target_frame,
    on="TargetDate",
    how="left",
    validate="one_to_one",
)

modelling_data = forecasting_panel.dropna(subset=target_columns).reset_index(drop=True)
assert modelling_data.shape[0] == 61
assert modelling_data["TargetDate"].is_monotonic_increasing
assert modelling_data["TargetDate"].reset_index(drop=True).equals(
    pd.Series(pd.date_range(
        modelling_data["TargetDate"].min(),
        modelling_data["TargetDate"].max(),
        freq="ME",
    ))
)

feature_columns = [column for column in D7.columns if column != "Date"]
X_all = modelling_data[feature_columns].to_numpy(dtype=float)
Y_all = modelling_data[target_columns].to_numpy(dtype=float)

initial_train_size = 36
minimum_selected_features = 5
stability_bootstrap_samples = 20
stability_selection_threshold = 0.60
elasticnet_alpha_grid = [0.001, 0.01, 0.05, 0.1, 0.5]
elasticnet_l1_ratio_grid = [0.2, 0.5, 0.8]
assert len(modelling_data) > initial_train_size
assert X_all.shape[0] == 61
assert X_all.shape[1] == len(feature_columns)
assert Y_all.shape == (61, 9)

def build_model(model_name, random_state, elasticnet_params=None):
    if model_name == "ElasticNet":
        if elasticnet_params is None:
            elasticnet_params = {"alpha": 0.05, "l1_ratio": 0.5}
        return ElasticNet(
            alpha=elasticnet_params["alpha"],
            l1_ratio=elasticnet_params["l1_ratio"],
            max_iter=10000,
            random_state=random_state,
        )
    if model_name == "RandomForest":
        return RandomForestRegressor(
            n_estimators=100,
            max_depth=3,
            min_samples_leaf=3,
            max_features="sqrt",
            random_state=random_state,
            n_jobs=1,
        )
    if model_name == "XGBoost":
        return XGBRegressor(
            n_estimators=75,
            max_depth=2,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=5.0,
            objective="reg:squarederror",
            random_state=random_state,
            n_jobs=1,
            verbosity=0,
        )
    if model_name == "LightGBM":
        return LGBMRegressor(
            n_estimators=75,
            num_leaves=7,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=5.0,
            random_state=random_state,
            n_jobs=1,
            verbosity=-1,
        )
    if model_name == "CatBoost":
        return CatBoostRegressor(
            iterations=75,
            depth=3,
            learning_rate=0.05,
            l2_leaf_reg=5.0,
            random_seed=random_state,
            verbose=False,
            thread_count=1,
            allow_writing_files=False,
        )
    raise ValueError(f"Unknown model: {model_name}")

def extract_selector_scores(fitted_selector, model_name):
    fitted_estimator = fitted_selector.estimator_
    if model_name == "ElasticNet":
        scores = np.abs(fitted_estimator.coef_)
    elif model_name == "CatBoost":
        scores = np.asarray(fitted_estimator.get_feature_importance(), dtype=float)
    else:
        scores = np.asarray(fitted_estimator.feature_importances_, dtype=float)
    return np.nan_to_num(scores, nan=0.0, posinf=0.0, neginf=0.0)

def fit_selector_once(X_train, y_train, model_name, random_state, elasticnet_params=None):
    selector_model = build_model(
        model_name,
        random_state=random_state,
        elasticnet_params=elasticnet_params,
    )
    if model_name == "CatBoost":
        selector = SelectFromModel(
            estimator=selector_model,
            importance_getter=lambda estimator: estimator.get_feature_importance(),
        )
    else:
        selector = SelectFromModel(estimator=selector_model)
    selector.fit(X_train, y_train)

    selected_mask = selector.get_support()
    selector_scores = extract_selector_scores(selector, model_name)
    return selected_mask, selector_scores

def tune_elasticnet_hyperparameters(X_train, y_train, random_state):
    n_splits = min(5, max(2, len(y_train) // 6))
    time_series_split = TimeSeriesSplit(n_splits=n_splits)
    best_score = np.inf
    best_params = {
        "alpha": elasticnet_alpha_grid[0],
        "l1_ratio": elasticnet_l1_ratio_grid[0],
    }

    for alpha in elasticnet_alpha_grid:
        for l1_ratio in elasticnet_l1_ratio_grid:
            fold_rmses = []
            for train_index, validation_index in time_series_split.split(X_train):
                X_train_fold = X_train[train_index]
                y_train_fold = y_train[train_index]
                X_validation_fold = X_train[validation_index]
                y_validation_fold = y_train[validation_index]

                fold_scaler = StandardScaler()
                X_train_fold_scaled = fold_scaler.fit_transform(X_train_fold)
                X_validation_fold_scaled = fold_scaler.transform(X_validation_fold)

                fold_model = ElasticNet(
                    alpha=alpha,
                    l1_ratio=l1_ratio,
                    max_iter=10000,
                    random_state=random_state,
                )
                fold_model.fit(X_train_fold_scaled, y_train_fold)
                validation_prediction = fold_model.predict(X_validation_fold_scaled)
                fold_rmses.append(
                    float(np.sqrt(mean_squared_error(y_validation_fold, validation_prediction)))
                )

            average_rmse = float(np.mean(fold_rmses))
            if average_rmse < best_score - 1e-12:
                best_score = average_rmse
                best_params = {"alpha": alpha, "l1_ratio": l1_ratio}

    return best_params

def fit_stability_selection_mask(
    X_train_raw,
    y_train,
    model_name,
    random_state_base,
    full_variance_support,
    elasticnet_params=None,
    bootstrap_samples=30,
    stability_threshold=0.60,
):
    candidate_indices = np.flatnonzero(full_variance_support)
    n_candidates = len(candidate_indices)
    assert n_candidates > 0

    selection_counts = np.zeros(n_candidates, dtype=float)
    importance_sums = np.zeros(n_candidates, dtype=float)
    importance_counts = np.zeros(n_candidates, dtype=float)

    bootstrap_rng = np.random.default_rng(random_state_base)

    for bootstrap_index in range(bootstrap_samples):
        sample_index = bootstrap_rng.integers(0, X_train_raw.shape[0], size=X_train_raw.shape[0])
        X_bootstrap_raw = X_train_raw[sample_index][:, candidate_indices]
        y_bootstrap = y_train[sample_index]

        bootstrap_variance_filter = VarianceThreshold(threshold=1e-12)
        X_bootstrap_variance = bootstrap_variance_filter.fit_transform(X_bootstrap_raw)
        if X_bootstrap_variance.shape[1] == 0:
            continue
        bootstrap_support = bootstrap_variance_filter.get_support()

        if model_name == "ElasticNet":
            bootstrap_selector_scaler = StandardScaler()
            X_bootstrap_for_selection = bootstrap_selector_scaler.fit_transform(
                X_bootstrap_variance
            )
        else:
            X_bootstrap_for_selection = X_bootstrap_variance

        selector_random_state = random_state_base + 10000 + bootstrap_index
        bootstrap_selected_mask, bootstrap_scores = fit_selector_once(
            X_bootstrap_for_selection,
            y_bootstrap,
            model_name=model_name,
            random_state=selector_random_state,
            elasticnet_params=elasticnet_params,
        )

        bootstrap_candidate_indices = np.flatnonzero(bootstrap_support)
        selection_counts[bootstrap_candidate_indices[bootstrap_selected_mask]] += 1.0
        importance_sums[bootstrap_candidate_indices] += bootstrap_scores
        importance_counts[bootstrap_candidate_indices] += 1.0

    selection_frequency = selection_counts / float(bootstrap_samples)
    selected_mask = selection_frequency >= stability_threshold

    minimum_required = min(minimum_selected_features, n_candidates)
    if selected_mask.sum() < minimum_required:
        average_importance = np.divide(
            importance_sums,
            importance_counts,
            out=np.zeros_like(importance_sums),
            where=importance_counts > 0,
        )
        top_indices = np.argsort(average_importance)[-minimum_required:]
        selected_mask = np.zeros(n_candidates, dtype=bool)
        selected_mask[top_indices] = True

    return selected_mask

model_names = ["ElasticNet", "RandomForest", "XGBoost", "LightGBM", "CatBoost"]
forecast_rows = []
importance_rows = []
selection_origins_by_feature_model = {}

for origin in range(initial_train_size, len(modelling_data)):
    X_train_raw = X_all[:origin]
    X_test_raw = X_all[origin:origin + 1]
    y_train_all = Y_all[:origin]
    target_date = modelling_data.loc[origin, "TargetDate"]
    feature_date = modelling_data.loc[origin, "Date"]
    origin_key = (feature_date, target_date)

    for sector_index, sector in enumerate(sector_columns):
        y_train = y_train_all[:, sector_index]
        actual = float(Y_all[origin, sector_index])

        variance_filter = VarianceThreshold(threshold=1e-12)
        X_train_variance = variance_filter.fit_transform(X_train_raw)
        X_test_variance = variance_filter.transform(X_test_raw)
        variance_support = variance_filter.get_support()
        variance_feature_names = np.asarray(feature_columns)[variance_support]

        for model_index, model_name in enumerate(model_names):
            random_state = 1000 + 100 * origin + 10 * sector_index + model_index
            elasticnet_params = None
            if model_name == "ElasticNet":
                elasticnet_params = tune_elasticnet_hyperparameters(
                    X_train_variance,
                    y_train,
                    random_state=random_state,
                )

            selected_mask = fit_stability_selection_mask(
                X_train_raw=X_train_raw,
                y_train=y_train,
                model_name=model_name,
                random_state_base=random_state,
                full_variance_support=variance_support,
                elasticnet_params=elasticnet_params,
                bootstrap_samples=stability_bootstrap_samples,
                stability_threshold=stability_selection_threshold,
            )

            X_train_selected_raw = X_train_variance[:, selected_mask]
            X_test_selected_raw = X_test_variance[:, selected_mask]
            selected_feature_names = variance_feature_names[selected_mask]
            minimum_required = min(minimum_selected_features, X_train_variance.shape[1])
            assert len(selected_feature_names) >= minimum_required

            if model_name == "ElasticNet":
                scaler = StandardScaler()
                X_train_selected = scaler.fit_transform(X_train_selected_raw)
                X_test_selected = scaler.transform(X_test_selected_raw)

                model = build_model(
                    model_name,
                    random_state=random_state,
                    elasticnet_params=elasticnet_params,
                )
                model.fit(X_train_selected, y_train)
                prediction = float(model.predict(X_test_selected)[0])
                importances = np.abs(model.coef_)
            else:
                model = build_model(model_name, random_state=random_state)
                model.fit(X_train_selected_raw, y_train)
                prediction = float(model.predict(X_test_selected_raw)[0])
                if model_name == "CatBoost":
                    importances = model.get_feature_importance()
                else:
                    importances = model.feature_importances_

            forecast_rows.append(
                {
                    "FeatureDate": feature_date,
                    "TargetDate": target_date,
                    "Sector": sector,
                    "Model": model_name,
                    "ActualExcessLogReturn": actual,
                    "PredictedExcessLogReturn": prediction,
                    "ForecastError": actual - prediction,
                }
            )
            importance_rows.extend(
                {
                    "FeatureDate": feature_date,
                    "TargetDate": target_date,
                    "Sector": sector,
                    "Model": model_name,
                    "Feature": feature,
                    "Importance": float(importance),
                }
                for feature, importance in zip(selected_feature_names, importances)
            )

            for feature in selected_feature_names:
                selection_origins_by_feature_model.setdefault((feature, model_name), set()).add(origin_key)

        forecast_rows.append(
            {
                "FeatureDate": feature_date,
                "TargetDate": target_date,
                "Sector": sector,
                "Model": "ZeroBenchmark",
                "ActualExcessLogReturn": actual,
                "PredictedExcessLogReturn": 0.0,
                "ForecastError": actual,
            }
        )

forecasts = pd.DataFrame(forecast_rows)
base_model_forecasts = forecasts[forecasts["Model"].isin(model_names)]
ensemble = (
    base_model_forecasts
    .groupby(["FeatureDate", "TargetDate", "Sector", "ActualExcessLogReturn"], as_index=False)
    .agg(PredictedExcessLogReturn=("PredictedExcessLogReturn", "median"))
)
ensemble["Model"] = "MedianEnsemble"
ensemble["ForecastError"] = (
    ensemble["ActualExcessLogReturn"] - ensemble["PredictedExcessLogReturn"]
)
forecasts = pd.concat((forecasts, ensemble[forecasts.columns]), ignore_index=True)
feature_importances = pd.DataFrame(importance_rows)

importance_group_keys = ["FeatureDate", "TargetDate", "Sector", "Model"]
importance_group_sums = feature_importances.groupby(importance_group_keys)["Importance"].transform("sum")
importance_group_counts = feature_importances.groupby(importance_group_keys)["Importance"].transform("size")
feature_importances["Importance"] = np.where(
    importance_group_sums > 0,
    feature_importances["Importance"] / importance_group_sums,
    1.0 / importance_group_counts,
 )

forecast_origin_count = int(modelling_data["TargetDate"].nunique())
feature_selection_frequency_rows = []
for model_name in model_names:
    for feature in feature_columns:
        selected_origin_count = len(
            selection_origins_by_feature_model.get((feature, model_name), set())
        )
        feature_selection_frequency_rows.append(
            {
                "Feature": feature,
                "Model": model_name,
                "SelectionFrequency": float(selected_origin_count / forecast_origin_count),
            }
        )
feature_selection_frequency = pd.DataFrame(feature_selection_frequency_rows)

assert forecasts.shape == (1575, 7)
assert feature_importances.shape[0] > 0
assert feature_importances.shape[1] == 6
assert forecasts.isna().sum().sum() == 0
assert feature_importances.isna().sum().sum() == 0
assert np.isfinite(forecasts[["ActualExcessLogReturn", "PredictedExcessLogReturn", "ForecastError"]].to_numpy()).all()
assert np.isfinite(feature_importances["Importance"].to_numpy()).all()
assert forecasts.groupby(["FeatureDate", "TargetDate", "Sector", "Model"]).size().eq(1).all()
assert forecasts.groupby(["TargetDate", "Sector"])["Model"].nunique().eq(7).all()
assert feature_importances["Feature"].isin(feature_columns).all()
assert feature_importances.groupby(importance_group_keys)["Feature"].nunique().eq(
    feature_importances.groupby(importance_group_keys).size()
).all()
assert np.allclose(
    feature_importances.groupby(importance_group_keys)["Importance"].sum().to_numpy(),
    1.0,
    atol=1e-10,
 )
assert feature_selection_frequency.shape == (len(feature_columns) * len(model_names), 3)
assert feature_selection_frequency["Feature"].isin(feature_columns).all()
assert feature_selection_frequency["Model"].isin(model_names).all()
assert feature_selection_frequency["SelectionFrequency"].between(0, 1).all()

metric_rows = []
for (model_name, sector), group in forecasts.groupby(["Model", "Sector"]):
    actual = group["ActualExcessLogReturn"].to_numpy()
    predicted = group["PredictedExcessLogReturn"].to_numpy()
    metric_rows.append(
        {
            "Model": model_name,
            "Sector": sector,
            "Observations": len(group),
            "RMSE": float(np.sqrt(mean_squared_error(actual, predicted))),
            "MAE": float(mean_absolute_error(actual, predicted)),
            "R2": float(r2_score(actual, predicted)),
            "DirectionalAccuracy": float(np.mean(np.sign(actual) == np.sign(predicted))),
        }
    )
forecast_metrics = pd.DataFrame(metric_rows)

assert forecast_metrics.shape == (63, 7)
assert forecast_metrics["Observations"].eq(25).all()
assert forecast_metrics.isna().sum().sum() == 0
assert np.isfinite(forecast_metrics.drop(columns=["Model", "Sector"]).to_numpy(dtype=float)).all()

D8 = forecasts.copy()
D8.to_csv("D8.csv", index=False)
forecast_metrics.to_csv("forecast_metrics.csv", index=False)
feature_importances.to_csv("feature_importances.csv", index=False)
feature_selection_frequency.to_csv("feature_selection_frequency.csv", index=False)

model_average_metrics = forecast_metrics.groupby("Model")[["RMSE", "MAE", "R2", "DirectionalAccuracy"]].mean()

module_8_validation = {
    "forecast_rows": int(len(D8)),
    "forecast_origins": int(D8["TargetDate"].nunique()),
    "models_per_sector_date": 7,
    "feature_importance_rows": int(len(feature_importances)),
    "metrics_shape": forecast_metrics.shape,
    "all_finite": True,
    "unique_forecast_keys": True,
    "mean_metrics": model_average_metrics.to_dict(),
    "look_ahead_bias": "At each origin, target-aligned data after the feature date are excluded; feature filters, selectors, scalers, and models are fit only on earlier rows.",
    "data_leakage": "No full-sample transformation, target encoding, hyperparameter tuning, or test-period model selection is used.",
}
module_8_validation

# Module 9

In [9]:
from scipy.optimize import minimize
from sklearn.covariance import LedoitWolf

D0 = pd.read_csv("D0.csv")
D1 = pd.read_csv("D1.csv")
D8 = pd.read_csv("D8.csv")
for frame in (D0, D1):
    frame["Date"] = pd.to_datetime(frame["Date"])
D8["FeatureDate"] = pd.to_datetime(D8["FeatureDate"])
D8["TargetDate"] = pd.to_datetime(D8["TargetDate"])

sector_columns = [
    "Materials", "Energy", "Financials", "Industrials", "Technology",
    "Consumer_Staples", "Utilities", "Healthcare", "Consumer_Discretionary",
]
excess_columns = [f"ExcessLogReturn_{sector}" for sector in sector_columns]
n_sectors = len(sector_columns)

assert D8.groupby(["FeatureDate", "TargetDate", "Sector", "Model"]).size().eq(1).all()
median_forecasts = D8[D8["Model"] == "MedianEnsemble"].copy()
assert median_forecasts.shape == (25 * n_sectors, 7)
assert median_forecasts.groupby(["FeatureDate", "TargetDate"])["Sector"].nunique().eq(n_sectors).all()

forecast_matrix = (
    median_forecasts
    .pivot(index=["FeatureDate", "TargetDate"], columns="Sector", values="PredictedExcessLogReturn")
    .reindex(columns=sector_columns)
    .reset_index()
)
assert forecast_matrix.shape == (25, 11)
assert forecast_matrix.isna().sum().sum() == 0

D0_prices = D0.set_index("Date")[sector_columns]
simple_sector_returns = D0_prices.pct_change().dropna()
risk_free_simple = D0.set_index("Date")["RiskFreeRate"] / 1200

D1_returns = D1.set_index("Date")[excess_columns]
covariance_window = 36
risk_aversion = 10.0
maximum_mean_variance_weight = 0.35

def mean_variance_weights(expected_returns, covariance):
    initial_weights = np.full(n_sectors, 1 / n_sectors)
    objective = lambda weights: -(
        expected_returns @ weights
        - 0.5 * risk_aversion * weights @ covariance @ weights
    )
    gradient = lambda weights: -expected_returns + risk_aversion * covariance @ weights
    solution = minimize(
        objective,
        initial_weights,
        jac=gradient,
        method="SLSQP",
        bounds=[(0.0, maximum_mean_variance_weight)] * n_sectors,
        constraints={"type": "eq", "fun": lambda weights: weights.sum() - 1},
        options={"ftol": 1e-12, "maxiter": 1000},
    )
    assert solution.success, solution.message
    return solution.x

def risk_parity_weights(covariance):
    equal_budget = 1 / n_sectors
    initial = np.full(n_sectors, 1 / n_sectors)
    objective = lambda x: (
        0.5 * x @ covariance @ x - equal_budget * np.log(x).sum()
    )
    gradient = lambda x: covariance @ x - equal_budget / x
    solution = minimize(
        objective,
        initial,
        jac=gradient,
        method="L-BFGS-B",
        bounds=[(1e-8, None)] * n_sectors,
        options={"ftol": 1e-14, "gtol": 1e-10, "maxiter": 5000},
    )
    assert solution.success, solution.message
    return solution.x / solution.x.sum()

weight_rows = []
return_rows = []

for _, forecast_row in forecast_matrix.iterrows():
    feature_date = forecast_row["FeatureDate"]
    target_date = forecast_row["TargetDate"]
    expected_returns = forecast_row[sector_columns].to_numpy(dtype=float)

    return_history = D1_returns.loc[D1_returns.index <= feature_date].tail(covariance_window)
    assert return_history.shape == (covariance_window, n_sectors)
    covariance = LedoitWolf().fit(return_history.to_numpy()).covariance_
    covariance = (covariance + covariance.T) / 2
    assert np.all(np.linalg.eigvalsh(covariance) > 0)

    strategy_weights = {
        "EqualWeight": np.full(n_sectors, 1 / n_sectors),
        "MeanVariance": mean_variance_weights(expected_returns, covariance),
        "RiskParity": risk_parity_weights(covariance),
    }

    realised_simple_returns = simple_sector_returns.loc[target_date].to_numpy(dtype=float)
    assert np.isfinite(realised_simple_returns).all()
    risk_free_return = float(risk_free_simple.loc[target_date])

    for strategy, weights in strategy_weights.items():
        portfolio_simple_return = float(weights @ realised_simple_returns)
        portfolio_excess_simple_return = portfolio_simple_return - risk_free_return
        portfolio_variance = float(weights @ covariance @ weights)
        marginal_risk = covariance @ weights
        risk_contributions = weights * marginal_risk
        normalized_risk_contributions = risk_contributions / risk_contributions.sum()

        for sector_index, sector in enumerate(sector_columns):
            weight_rows.append(
                {
                    "FeatureDate": feature_date,
                    "TargetDate": target_date,
                    "Strategy": strategy,
                    "Sector": sector,
                    "Weight": float(weights[sector_index]),
                    "ForecastExcessLogReturn": float(expected_returns[sector_index]),
                    "RiskContribution": float(normalized_risk_contributions[sector_index]),
                }
            )

        return_rows.append(
            {
                "FeatureDate": feature_date,
                "TargetDate": target_date,
                "Strategy": strategy,
                "PortfolioSimpleReturn": portfolio_simple_return,
                "PortfolioExcessSimpleReturn": portfolio_excess_simple_return,
                "RiskFreeSimpleReturn": risk_free_return,
                "ExAnteVariance": portfolio_variance,
            }
        )

D9 = pd.DataFrame(weight_rows)
portfolio_returns = pd.DataFrame(return_rows)

assert D9.shape == (25 * 3 * n_sectors, 7)
assert portfolio_returns.shape == (25 * 3, 7)
assert D9.isna().sum().sum() == 0
assert portfolio_returns.isna().sum().sum() == 0
assert np.isfinite(D9[["Weight", "ForecastExcessLogReturn", "RiskContribution"]].to_numpy()).all()
assert np.isfinite(portfolio_returns.drop(columns=["FeatureDate", "TargetDate", "Strategy"]).to_numpy()).all()
assert D9.groupby(["TargetDate", "Strategy"])["Weight"].sum().pipe(
    lambda values: np.allclose(values.to_numpy(), 1.0)
)
assert D9["Weight"].ge(-1e-10).all()
assert D9[D9["Strategy"] == "MeanVariance"]["Weight"].le(
    maximum_mean_variance_weight + 1e-8
).all()
assert np.allclose(
    D9.groupby(["TargetDate", "Strategy"])["RiskContribution"].sum().to_numpy(),
    1.0,
)
risk_parity_contributions = (
    D9[D9["Strategy"] == "RiskParity"]
    .groupby("TargetDate")["RiskContribution"]
    .apply(lambda values: np.max(np.abs(values.to_numpy() - 1 / n_sectors)))
)
assert risk_parity_contributions.max() < 1e-6
assert portfolio_returns.groupby(["TargetDate", "Strategy"]).size().eq(1).all()
assert portfolio_returns.groupby("Strategy")["TargetDate"].nunique().eq(25).all()

D9.to_csv("D9.csv", index=False)
portfolio_returns.to_csv("portfolio_returns.csv", index=False)

module_9_validation = {
    "weight_shape": D9.shape,
    "portfolio_return_shape": portfolio_returns.shape,
    "date_range": f"{portfolio_returns['TargetDate'].min().date()} to {portfolio_returns['TargetDate'].max().date()}",
    "strategies": sorted(portfolio_returns["Strategy"].unique().tolist()),
    "long_only_weights": True,
    "weights_sum_to_one": True,
    "mean_variance_weight_cap": maximum_mean_variance_weight,
    "risk_parity_max_budget_deviation": float(risk_parity_contributions.max()),
    "finite_covariances": True,
    "look_ahead_bias": "Each covariance matrix uses only the trailing 36 D1 returns available on the feature date; weights use forecasts generated before the target month.",
    "data_leakage": "Realised target-month returns are used only after weights are fixed to evaluate portfolio outcomes.",
}
module_9_validation



{'weight_shape': (675, 7),
 'portfolio_return_shape': (75, 7),
 'date_range': '2024-02-29 to 2026-02-28',
 'strategies': ['EqualWeight', 'MeanVariance', 'RiskParity'],
 'long_only_weights': True,
 'weights_sum_to_one': True,
 'mean_variance_weight_cap': 0.35,
 'risk_parity_max_budget_deviation': 4.4953387540180856e-09,
 'finite_covariances': True,
 'look_ahead_bias': 'Each covariance matrix uses only the trailing 36 D1 returns available on the feature date; weights use forecasts generated before the target month.',
 'data_leakage': 'Realised target-month returns are used only after weights are fixed to evaluate portfolio outcomes.'}

# Module 10

In [10]:
D9 = pd.read_csv("D9.csv")
portfolio_returns = pd.read_csv("portfolio_returns.csv")
D9["FeatureDate"] = pd.to_datetime(D9["FeatureDate"])
D9["TargetDate"] = pd.to_datetime(D9["TargetDate"])
portfolio_returns["FeatureDate"] = pd.to_datetime(portfolio_returns["FeatureDate"])
portfolio_returns["TargetDate"] = pd.to_datetime(portfolio_returns["TargetDate"])

sector_columns = [
    "Materials", "Energy", "Financials", "Industrials", "Technology",
    "Consumer_Staples", "Utilities", "Healthcare", "Consumer_Discretionary",
]
strategies = ["EqualWeight", "MeanVariance", "RiskParity"]

assert D9.shape == (675, 7)
assert portfolio_returns.shape == (75, 7)
assert D9.groupby(["TargetDate", "Strategy"])["Sector"].nunique().eq(9).all()
assert D9.groupby(["TargetDate", "Strategy"])["Weight"].sum().pipe(
    lambda weights: np.allclose(weights.to_numpy(), 1.0)
)
assert portfolio_returns.groupby(["TargetDate", "Strategy"]).size().eq(1).all()
assert portfolio_returns.groupby("Strategy")["TargetDate"].nunique().eq(25).all()

weight_panel = (
    D9
    .pivot(index=["TargetDate", "Strategy"], columns="Sector", values="Weight")
    .reindex(columns=sector_columns)
    .reset_index()
    .sort_values(["Strategy", "TargetDate"])
    .reset_index(drop=True)
)

turnover_rows = []
for strategy, group in weight_panel.groupby("Strategy", sort=False):
    weights = group[sector_columns].to_numpy(dtype=float)
    turnover = np.zeros(len(group))
    turnover[1:] = 0.5 * np.abs(np.diff(weights, axis=0)).sum(axis=1)
    turnover_rows.extend(
        {
            "TargetDate": target_date,
            "Strategy": strategy,
            "OneWayTurnover": float(value),
        }
        for target_date, value in zip(group["TargetDate"], turnover)
    )
turnover = pd.DataFrame(turnover_rows)

D10 = (
    portfolio_returns
    .merge(turnover, on=["TargetDate", "Strategy"], how="inner", validate="one_to_one")
    .sort_values(["Strategy", "TargetDate"])
    .reset_index(drop=True)
)

backtest_rows = []
for strategy, group in D10.groupby("Strategy", sort=False):
    group = group.sort_values("TargetDate").copy()
    simple_returns = group["PortfolioSimpleReturn"].to_numpy(dtype=float)
    excess_returns = group["PortfolioExcessSimpleReturn"].to_numpy(dtype=float)
    benchmark_returns = D10[
        (D10["Strategy"] == "EqualWeight")
        & (D10["TargetDate"].isin(group["TargetDate"]))
    ].sort_values("TargetDate")["PortfolioSimpleReturn"].to_numpy(dtype=float)

    cumulative_wealth = np.cumprod(1 + simple_returns)
    running_peak = np.maximum.accumulate(cumulative_wealth)
    drawdown = cumulative_wealth / running_peak - 1

    group["CumulativeWealth"] = cumulative_wealth
    group["Drawdown"] = drawdown
    D10.loc[group.index, ["CumulativeWealth", "Drawdown"]] = group[
        ["CumulativeWealth", "Drawdown"]
    ]

    observations = len(group)
    annual_return = float(cumulative_wealth[-1] ** (12 / observations) - 1)
    annual_volatility = float(np.std(simple_returns, ddof=1) * np.sqrt(12))
    sharpe_ratio = float(
        np.mean(excess_returns) / np.std(excess_returns, ddof=1) * np.sqrt(12)
    )
    downside_returns = np.minimum(excess_returns, 0.0)
    downside_deviation = float(np.sqrt(np.mean(downside_returns ** 2)))
    sortino_ratio = float(
        np.mean(excess_returns) / downside_deviation * np.sqrt(12)
    ) if downside_deviation > 0 else 0.0
    maximum_drawdown = float(drawdown.min())
    calmar_ratio = float(
        annual_return / abs(maximum_drawdown)
    ) if maximum_drawdown < 0 else 0.0
    active_returns = simple_returns - benchmark_returns
    active_volatility = float(np.std(active_returns, ddof=1))
    information_ratio = float(
        np.mean(active_returns) / active_volatility * np.sqrt(12)
    ) if active_volatility > 1e-12 else 0.0

    backtest_rows.append(
        {
            "Strategy": strategy,
            "Observations": observations,
            "AnnualReturn": annual_return,
            "AnnualVolatility": annual_volatility,
            "SharpeRatio": sharpe_ratio,
            "SortinoRatio": sortino_ratio,
            "CalmarRatio": calmar_ratio,
            "MaximumDrawdown": maximum_drawdown,
            "AverageMonthlyTurnover": float(group["OneWayTurnover"].mean()),
            "HitRatio": float(np.mean(excess_returns > 0)),
            "InformationRatioVsEqualWeight": information_ratio,
        }
    )

backtest_metrics = pd.DataFrame(backtest_rows).sort_values("Strategy").reset_index(drop=True)

assert D10.shape == (75, 10)
assert D10["TargetDate"].duplicated().sum() == 50
assert D10.groupby("Strategy")["TargetDate"].apply(
    lambda dates: dates.reset_index(drop=True).equals(
        pd.Series(pd.date_range(dates.min(), dates.max(), freq="ME"))
    )
).all()
assert D10.isna().sum().sum() == 0
assert np.isfinite(
    D10.drop(columns=["FeatureDate", "TargetDate", "Strategy"]).to_numpy(dtype=float)
).all()
assert backtest_metrics.shape == (3, 11)
assert backtest_metrics.isna().sum().sum() == 0
assert np.isfinite(backtest_metrics.drop(columns="Strategy").to_numpy(dtype=float)).all()
assert backtest_metrics["Observations"].eq(25).all()
assert backtest_metrics["MaximumDrawdown"].le(0).all()
assert backtest_metrics["AverageMonthlyTurnover"].ge(0).all()

D10.to_csv("D10.csv", index=False)
backtest_metrics.to_csv("backtest_metrics.csv", index=False)

module_10_validation = {
    "backtest_path_shape": D10.shape,
    "metrics_shape": backtest_metrics.shape,
    "date_range": f"{D10['TargetDate'].min().date()} to {D10['TargetDate'].max().date()}",
    "strategies": backtest_metrics["Strategy"].tolist(),
    "monthly_continuity_per_strategy": True,
    "finite_values": True,
    "weight_turnover_aligned": True,
    "gross_returns_only": True,
    "look_ahead_bias": "Weights were fixed in Module 9 using information through each feature date; this module evaluates only subsequent realised target-month returns.",
    "data_leakage": "No realised return is used to modify a previously fixed portfolio weight or model forecast.",
}
module_10_validation



{'backtest_path_shape': (75, 10),
 'metrics_shape': (3, 11),
 'date_range': '2024-02-29 to 2026-02-28',
 'strategies': ['EqualWeight', 'MeanVariance', 'RiskParity'],
 'monthly_continuity_per_strategy': True,
 'finite_values': True,
 'weight_turnover_aligned': True,
 'gross_returns_only': True,
 'look_ahead_bias': 'Weights were fixed in Module 9 using information through each feature date; this module evaluates only subsequent realised target-month returns.',
 'data_leakage': 'No realised return is used to modify a previously fixed portfolio weight or model forecast.'}